In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from math import sqrt

import seaborn as sns
sns.set_theme("notebook",style="dark")
gratio = (1+sqrt(5))/2

In [ ]:
import torch
from qgsw.logging.core import getLogger, setup_root_logger
from qgsw.specs import defaults

from __future__ import annotations

import os
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import torch
import xarray as xr
from scipy.ndimage import gaussian_filter

from qgsw.configs.core import Configuration
from qgsw.decomposition.core import build_basis_from_params_dict
from qgsw.eNATL60.fields_computations import compute_stream_function_ssh_only
from qgsw.eNATL60.interpolation import (
    build_regridder,
    compute_lonlat_from_regular_xy_grid,
    lonlat_to_xy,
)
from qgsw.eNATL60.loading import load_datasets
from qgsw.eNATL60.var_keys import (
    LATITUDE,
    LONGITUDE,
    SSH,
    STREAMFUNCTION,
    TIME,
)
from qgsw.logging.utils import box, step
from qgsw.masks import Masks
from qgsw.models.qg.psiq.core import QGPSIQ
from qgsw.physics.constants import EARTH_ANGULAR_ROTATION, EARTH_RADIUS
from qgsw.physics.coriolis.beta_plane import BetaPlane
from qgsw.solver.boundary_conditions.base import Boundaries
from qgsw.spatial.core.discretization import (
    SpaceDiscretization2D,
)
from qgsw.spatial.core.grid import Grid2D
from qgsw.utils.storage import get_path_from_env
from qgsw.utils.tensor_dict import change_specs


torch.backends.cudnn.deterministic = True

ROOT_PATH = Path(os.getcwd()).parent


specs=defaults.get()

logger=getLogger(__name__)
setup_root_logger(1)

In [ ]:
dt = 7200
n_file_per_cycle = 20
n_steps_per_cyle = 240
separation = 0
comparison_interval = 1
n_cycles=4

### Load filenames

In [ ]:
from qgsw.eNATL60 import seasons
from qgsw.eNATL60.loading import retrieve_dates, sort_files_by_dates


data_folder = get_path_from_env(key="eNATL60_FOLDER")
files = list((data_folder / "MEANDERS" / "gridT").glob("*.nc"))

files = sort_files_by_dates(*files)

season_dict = {
    "summer": seasons.SUMMER,
    "autumn": seasons.AUTUMN,
    "winter": seasons.WINTER,
    "spring": seasons.SPRING,
}

### Dataset formatting func

In [ ]:
def format_ds(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "sossheig": SSH,
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])

### Build space

In [ ]:
ds_init = load_datasets(files[0], format_func=format_ds)

dx = dy = 10000
lons, lats = compute_lonlat_from_regular_xy_grid(
    ds_init[LONGITUDE],
    ds_init[LATITUDE],
    dx=dx,
    dy=dy,
)
xs, ys = lonlat_to_xy(lons, lats)

### Compute β-plane parameters

lat0 = (lats.max() + lats.min()) / 2
beta_plane = BetaPlane(
    f0=2 * EARTH_ANGULAR_ROTATION * np.sin(lat0),
    beta=2 * EARTH_ANGULAR_ROTATION * np.cos(lat0) / EARTH_RADIUS,
)

### Build regridder

psi_regridder = build_regridder(ds_init, lons, lats)


## Areas
nx, ny = lats.shape
xx = torch.tensor(xs.round(), **specs)
space_2d = SpaceDiscretization2D.from_psi_grid(
    Grid2D(
        x=xx - xx[0, :],
        y=torch.tensor(ys.round(), **specs),
    )
)
### Boundaries offset

b = 4

space_interior = space_2d.slice(
    b,
    space_2d.psi.xy.x.shape[0] - b,
    b,
    space_2d.psi.xy.x.shape[1] - b,
)

nx = space_interior.nx
ny = space_interior.ny
dx = space_interior.dx
dy = space_interior.dy

In [ ]:
space_2d.psi.xy.x.shape

In [ ]:
nx,ny

### Parameters

In [ ]:
ROOT_PATH = Path(os.getcwd()).parent

# Simulation parameters

config = Configuration.from_toml(ROOT_PATH.joinpath("config/enatl60.toml"))

H = config.model.h
H1,H2,H3 = H
g_prime = config.model.g_prime
g1,g2,g3= g_prime
# beta_plane = config.physics.beta_plane
bottom_drag_coef = config.physics.bottom_drag_coefficient
slip_coef = config.physics.slip_coef

y_w = space_2d.q.xy.y[0, :].unsqueeze(0)
y0 = 0.5 * space_interior.ly
beta_effect = beta_plane.beta * (y_w - y0)

In [ ]:
def xy_to_lonlat(x:np.ndarray, y:np.ndarray, lat0:float) -> tuple[np.ndarray, np.ndarray]:
    lats = (y/EARTH_RADIUS)+lat0
    lons = x/EARTH_RADIUS/np.cos(lats)
    return lons,lats

In [ ]:
def format_dsu(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "sozocrtx": "u",
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])
def format_dsv(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "somecrty": "v",
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])

    
space_no_xoffset=SpaceDiscretization2D.from_psi_grid(
    Grid2D(
        x=torch.tensor(xs.round(),**specs),
        y=torch.tensor(ys.round(), **specs),
    )
)

ux,uy  =space_no_xoffset.u.xy
ux=ux.cpu().numpy()
uy=uy.cpu().numpy()

lonsu, latsu = xy_to_lonlat(ux,uy,lat0)

vx,vy  =space_no_xoffset.v.xy
vx=vx.cpu().numpy()
vy=vy.cpu().numpy()

lonsv, latsv = xy_to_lonlat(vx,vy,lat0)

ufiles = list(
    (data_folder/"MEANDERS"/"gridU").glob("*.nc"),
)
ufiles = sort_files_by_dates(*ufiles)

dsu_init = load_datasets(ufiles[0], format_func=format_dsu)

u_regridder = build_regridder(dsu_init, lonsu, latsu)

vfiles = list(
    (data_folder/"MEANDERS"/"gridV").glob("*.nc"),
)
vfiles = sort_files_by_dates(*vfiles)

dsv_init = load_datasets(vfiles[0], format_func=format_dsv)

v_regridder = build_regridder(dsv_init, lonsv, latsv)

def load_cycle_uv_data(ufiles:np.ndarray, vfiles:np.ndarray,c:int) -> tuple[list[torch.Tensor],list[torch.Tensor], list[torch.Tensor]]:

    start_cycle = c * n_file_per_cycle + c * separation
    end_cycle = (c + 1) * n_file_per_cycle + c * separation

    if end_cycle > len(ufiles):
        msg = f"Not enough ufiles to perform cycle {c} and above."
        logger.warning(msg)
        return

    ufiles_for_cycle = ufiles[start_cycle:end_cycle]

    start_cycle = c * n_file_per_cycle + c * separation
    end_cycle = (c + 1) * n_file_per_cycle + c * separation

    if end_cycle > len(vfiles):
        msg = f"Not enough vfiles to perform cycle {c} and above."
        logger.warning(msg)
        return

    vfiles_for_cycle = vfiles[start_cycle:end_cycle]

    dsu = load_datasets(*ufiles_for_cycle, format_func=format_dsu)

    with logger.timeit("Interpolating u"):
        regridded_u: xr.DataArray = u_regridder(dsu["u"])
        dsu_interp = xr.Dataset(
            {
                LONGITUDE: (["i", "j"], lonsu),
                LATITUDE: (["i", "j"], latsu),
                "u": ([TIME, "i", "j"], regridded_u.data),
            },
            regridded_u.coords,
        )
        dsu_interp = dsu_interp.set_coords([LONGITUDE, LATITUDE])
        dsu_interp = dsu_interp.load()

    files_for_cycle = vfiles[c * n_file_per_cycle : (c + 1) * n_file_per_cycle]
    dsv = load_datasets(*vfiles_for_cycle, format_func=format_dsv)

    with logger.timeit("Interpolating v"):
        regridded_v: xr.DataArray = v_regridder(dsv["v"])
        dsv_interp = xr.Dataset(
            {
                LONGITUDE: (["i", "j"], lonsv),
                LATITUDE: (["i", "j"], latsv),
                "v": ([TIME, "i", "j"], regridded_v.data),
            },
            regridded_v.coords,
        )
        dsv_interp = dsv_interp.set_coords([LONGITUDE, LATITUDE])
        dsv_interp = dsv_interp.load()
    

    u_refs = [
        torch.tensor(u, **specs).unsqueeze(0).unsqueeze(0)
        for u in dsu_interp["u"].to_numpy()
    ]
    v_refs = [
        torch.tensor(v, **specs).unsqueeze(0).unsqueeze(0)
        for v in dsv_interp["v"].to_numpy()
    ]

    return u_refs, v_refs

In [ ]:
from qgsw.eNATL60.fields_computations import compute_streamfunction_with_atmospheric_pressure
from qgsw.eNATL60.forcing import load_era_interim, slice_space, slice_time


def load_cycle_data_atmp(files:np.ndarray, c:int) -> tuple[list[torch.Tensor],list[torch.Tensor], list[torch.Tensor], xr.DataArray]:
    start_cycle = c * n_file_per_cycle + c * separation
    end_cycle = (c + 1) * n_file_per_cycle + c * separation

    if end_cycle > len(files):
        msg = f"Not enough files to perform cycle {c} and above."
        logger.warning(msg)
        return

    files_for_cycle = files[start_cycle:end_cycle]

    ds = load_datasets(*files_for_cycle.tolist(), format_func=format_ds)
    dates = retrieve_dates(*files_for_cycle.tolist())
    years = dates.year.unique().to_list()
    if dates.min().month == 1 and dates.min().day == 1:
        years.insert(0, dates.min().year - 1)
    msg = f"Loading data for years: {', '.join([str(y) for y in years])}"
    logger.info(msg)
    ds_era = load_era_interim(data_folder / "misc", *years)

    ds_era = slice_time(ds_era, ds[TIME])
    ds_era = slice_space(ds_era, ds[LONGITUDE], ds[LATITUDE])

    msg = f"Cycle {step(c + 1, n_cycles)}: eNATL60 data loaded."
    logger.info(box(msg, style="round"))

    ds[STREAMFUNCTION] = compute_streamfunction_with_atmospheric_pressure(
        ds,
        ds_era,
        config.physics.rho,
        g_prime[0].item(),
        remove_avgs=True,
    )

    with logger.timeit("Filtering stream function"):
        ds["psi_filt"] = xr.apply_ufunc(
            gaussian_filter,
            ds[STREAMFUNCTION].load(),
            kwargs={"sigma": 16},
            input_core_dims=[["i", "j"]],
            output_core_dims=[["i", "j"]],
            vectorize=True,
        )

    with logger.timeit("Interpolating stream function"):
        regridded_psi: xr.DataArray = psi_regridder(ds[STREAMFUNCTION])
        regridded_psi_filt: xr.DataArray = psi_regridder(ds["psi_filt"])
        ds_interp = xr.Dataset(
            {
                LONGITUDE: (["i", "j"], lons),
                LATITUDE: (["i", "j"], lats),
                STREAMFUNCTION: ([TIME, "i", "j"], regridded_psi.data),
                "psi_filt": ([TIME, "i", "j"], regridded_psi_filt.data),
            },
            regridded_psi_filt.coords,
        )
        ds_interp = ds_interp.set_coords([LONGITUDE, LATITUDE])
        ds_interp = ds_interp.load()

    psis_ref = [
        torch.tensor(p, **specs).unsqueeze(0).unsqueeze(0)/beta_plane.f0
        for p in ds_interp[STREAMFUNCTION].to_numpy()
    ]
    psis_filt = [
        torch.tensor(p, **specs).unsqueeze(0).unsqueeze(0)/beta_plane.f0
        for p in ds_interp["psi_filt"].to_numpy()
    ]
    t0 = ds_interp[TIME][0]
    times = (ds_interp[TIME] - t0).dt.total_seconds().to_numpy()
    times = torch.tensor(times, **specs)

    return psis_ref,psis_filt,times, ds[STREAMFUNCTION][0]

### Vorticity

In [ ]:
from qgsw.eNATL60.var_keys import VORTICITY


data_folder = get_path_from_env(key="eNATL60_FOLDER")
vort_files = list(
    (data_folder/"MEANDERS"/"gridVort").glob("*.nc"),
)
vort_files = sort_files_by_dates(*vort_files)

def format_dvort(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "vorticity": VORTICITY,
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])

def load_cycle_vorticity_data(files:np.ndarray, c:int) -> list[torch.Tensor]:
    start_cycle = c * n_file_per_cycle + c * separation
    end_cycle = (c + 1) * n_file_per_cycle + c * separation

    if end_cycle > len(files):
        msg = f"Not enough files to perform cycle {c} and above."
        logger.warning(msg)
        return

    files_for_cycle = files[start_cycle:end_cycle]

    dvort = load_datasets(*files_for_cycle, format_func=format_dvort)

    ds_lon = dvort[LONGITUDE]
    ds_lat=dvort[LATITUDE]

    lon_slice = (ds_lon >= np.rad2deg(lons[(b+1):-(b+1),(b+1):-(b+1)].min())) & (ds_lon <= np.rad2deg(lons[(b+1):-(b+1),(b+1):-(b+1)].max()))
    lat_slice = (ds_lat >= np.rad2deg(lats[(b+1):-(b+1),(b+1):-(b+1)].min())) & (ds_lat <= np.rad2deg(lats[(b+1):-(b+1),(b+1):-(b+1)].max()))

    Is,Js = np.meshgrid(dvort.i.values,dvort.j.values,indexing="ij")

    I_slice = Is[lon_slice&lat_slice]
    J_slice = Js[lon_slice&lat_slice]

    imin,imax = I_slice.min(),I_slice.max()
    jmin,jmax = J_slice.min(),J_slice.max()

    dvort = dvort.sel(i=slice(imin,imax+1),j=slice(jmin,jmax+1))

    vorticities = [
        torch.tensor(vort, **specs).unsqueeze(0).unsqueeze(0)/beta_plane.f0
        for vort in dvort[VORTICITY].to_numpy()
    ]
    return vorticities

def load_cycle_filt_vorticity_data(c:int) -> list[torch.Tensor]:
    if (c + 1) * n_file_per_cycle > len(vort_files):
        msg = f"Not enough files to read for cycle {c} and above."
        logger.warning(msg)
        raise ValueError


    files_for_cycle = vort_files[c * n_file_per_cycle : (c + 1) * n_file_per_cycle]

    dvort = load_datasets(*files_for_cycle, format_func=format_dvort)

    dvort["vort_filt"] = xr.apply_ufunc(
            gaussian_filter,
            dvort[VORTICITY].load(),
            kwargs={"sigma": 16},
            input_core_dims=[["i", "j"]],
            output_core_dims=[["i", "j"]],
            vectorize=True,
        )

    ds_lon = dvort[LONGITUDE]
    ds_lat=dvort[LATITUDE]

    lon_slice = (ds_lon >= np.rad2deg(lons[(b+1):-(b+1),(b+1):-(b+1)].min())) & (ds_lon <= np.rad2deg(lons[(b+1):-(b+1),(b+1):-(b+1)].max()))
    lat_slice = (ds_lat >= np.rad2deg(lats[(b+1):-(b+1),(b+1):-(b+1)].min())) & (ds_lat <= np.rad2deg(lats[(b+1):-(b+1),(b+1):-(b+1)].max()))

    Is,Js = np.meshgrid(dvort.i.values,dvort.j.values,indexing="ij")

    I_slice = Is[lon_slice&lat_slice]
    J_slice = Js[lon_slice&lat_slice]

    imin,imax = I_slice.min(),I_slice.max()
    jmin,jmax = J_slice.min(),J_slice.max()

    dvort = dvort.sel(i=slice(imin,imax+1),j=slice(jmin,jmax+1))

    vorticities = [
        torch.tensor(vort, **specs).unsqueeze(0).unsqueeze(0)/beta_plane.f0
        for vort in dvort["vort_filt"].to_numpy()
    ]
    return vorticities

### Wind

In [ ]:
def compute_drag_coef(wind_magnitude: torch.Tensor) -> torch.Tensor:
    """Compute drag coefficient.

    Based on formula from 'Diurnal to decadal global forcing for ocean and
    sea-ice models: the data sets and flux climatologies'
    by Large and Yeager (2004)

    Arbitrary threshold of 0.5 added to prevent error from null velocities.
    """
    threshold = torch.tensor(0.5, **specs)
    return 1e-3 * (
        0.142
        + 2.7 / torch.maximum(wind_magnitude, threshold)
        + wind_magnitude / 13.09
    )

### Error

In [ ]:
from qgsw.solver.finite_diff import grad_perp


def rmse(f: torch.Tensor, f_ref: torch.Tensor) -> float:
    """RMSE."""
    return ((f - f_ref).square().mean() / f_ref.square().mean()).sqrt()

def uv_rmse(psi:torch.Tensor,u_ref:torch.Tensor, v_ref:torch.Tensor) -> torch.Tensor:
    """Gradient RMSE."""
    u,v = grad_perp(psi)
    
    u/=dy
    v/=dx


    return ((u-u_ref).square().mean()+(v-v_ref).square().mean()).sqrt() / (u_ref.square().mean()+v_ref.square().mean()).sqrt()

### Boundary conditions

In [ ]:
def extract_psi_bc(psi: torch.Tensor) -> Boundaries:
    """Extract psi."""
    return Boundaries.extract(psi, b, -b - 1, b, -b - 1, 2)

## Model Wrappers

In [ ]:
from collections.abc import Callable
from pathlib import Path
from typing import TypeVar

from matplotlib import pyplot as plt
import numpy as np
from torch import Tensor

from qgsw.analysis.qg_model import ModelWrapper, ModelsManager
from qgsw.models.qg.psiq.core import QGPSIQCore
from qgsw.spatial.core.discretization import SpaceDiscretization2D

T = TypeVar("T", bound=QGPSIQCore)

class ModelWrapperOBC(ModelWrapper[T]):
    results_paths = Path("../output/g5k/param_optim")

    losses:dict[str,list[list[torch.Tensor]]] = {}
    uv10_to_uvsurf = torch.eye(2,**specs)
    
    show=True
    remember_psis = False

    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.losses = {
            "rmse": [],
            "uv_rmse": [],
            "psi_filt_rmse": [],
        }
    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor, sf0_da:xr.DataArray) -> None:
        super().setup(psis, times, beta_effect_w)
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

    def _set_params(self) -> None:
        space = self.model.space
        self.model.masks = Masks.empty_tensor(
            space.nx,
            space.ny,
            device=specs["device"],
        )
        self.model.bottom_drag_coef = 0
        self.model.wide = True
        self.model.slip_coef = slip_coef
        self.model.dt = dt
        
    def load(self)-> dict:
        file = self.results_paths.joinpath(self.prefix+".pt")
        return torch.load(file)
    def new_cycle(self) -> None:
        super().new_cycle()
        for loss in self.losses.values():
            loss.append([])
        if self.remember_psis:
            self.psis = []
            
    def add_loss(self, loss_value:float,loss_name:str) -> None:
        self.losses[loss_name][-1].append(loss_value)
        
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        if not self.show:
            return
        if ax is None:
            ax = plt.gca()
        cycles = [cycle] if cycle is not None else list(range(len(self.losses[loss_name])))
        time_offset= 0
        for i,c in enumerate(cycles):
            times = self.model.dt*np.arange(len(self.losses[loss_name][c]))/3600/24 + time_offset
            time_offset = times[-1] + self.model.dt/3600/24
            loss =  np.array(self.losses[loss_name][c])
            kwargs = self.plot_kwargs.copy()
            if i!= 0:
                kwargs.pop("label")
            ax.plot(times, loss, **kwargs)
        ax.set_xlabel("Time [day]")
    
    def compute_windstress(self, uv10:torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        usurf, vsurf = torch.einsum(
            "lm,mxy->lxy",
            self.uv10_to_uvsurf,
            uv10,
        )

        u_mags = torch.sqrt(uv10.square().sum(dim=0))

        Cd = compute_drag_coef(u_mags)

        rho_air = 1.225
        rho_water = config.physics.rho

        bulk_coef = Cd * rho_air / rho_water

        tauxs = bulk_coef * u_mags * usurf
        tauys = bulk_coef * u_mags * vsurf

        tauxs_i = (tauxs[1:, :] + tauxs[:-1, :]) / 2
        tauys_i = (tauys[:, 1:] + tauys[:, :-1]) / 2
        return tauxs_i, tauys_i
    
    def step(self) -> None:
        super().step()
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

        
            
        
M = TypeVar("M", bound=ModelWrapperOBC[QGPSIQCore])

class ModelsManagerOBC(ModelsManager[M]):

    loss_fn: dict[str, Callable[[torch.Tensor,torch.Tensor], torch.Tensor]]= {
        "rmse":rmse,
    }

    losses = list(loss_fn.keys())

    def remember_psis(self, mode:bool=True)-> None:
        for mw in self.model_wrappers:
            mw.remember_psis = mode

    def compute_loss(self, psi_ref:torch.Tensor) -> None:
        for loss_name in self.losses:
            self.loop_over_models(
                lambda mw: mw.add_loss(self.loss_fn[loss_name](mw.model.psi[0,0],psi_ref[0,0]).cpu().item(),loss_name)
            )

    def compute_uv_losses(self,u_ref:torch.Tensor, v_ref:torch.Tensor) -> None:
        self.loop_over_models(
            lambda mw: mw.add_loss(uv_rmse(mw.model.psi[0,0],u_ref[0,0],v_ref[0,0]).cpu().item(),"uv_rmse")
        )

    def compute_psi_filt_loss(self,psi_filt_ref:torch.Tensor) -> None:
        self.loop_over_models(
            lambda mw: mw.add_loss(rmse(mw.model.psi[0,0],psi_filt_ref[0,0]).cpu().item(),"psi_filt_rmse")
        )
    
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        self.loop_over_models(lambda mw: mw.plot_loss(loss_name=loss_name,ax=ax,cycle=cycle))
    
    def build_wind_forcing(self, uv10:torch.Tensor) -> None:
        self.loop_over_models(
            lambda mw: mw.set_wind_forcing(*mw.compute_windstress(uv10))
        )

    def setup(
        self,
        psis: list[torch.Tensor],
        times: list[torch.Tensor],
        beta_effect_w: torch.Tensor,
        sf0_da:xr.DataArray,
    ) -> None:
        """Setup all models."""
        self.loop_over_models(lambda mw: mw.setup(psis, times, beta_effect_w,sf0_da))

### Reduced Gravity

In [ ]:
from torch import Tensor
from qgsw.solver.finite_diff import laplacian
from qgsw.spatial.core.discretization import SpaceDiscretization2D
from qgsw.spatial.core.grid_conversion import interpolate
from qgsw.utils.interpolation import QuadraticInterpolation
from qgsw.utils.reshaping import crop


class ReducedGravity(ModelWrapperOBC[QGPSIQ]):
    prefix = None
    color = "black"
    label="Reduced Gravity"
    sigma = 14
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQ(
            space_2d=space_2d,
            H=H[:1],
            beta_plane=beta_plane,
            g_prime=g_prime[:1]*g_prime[1:2]/(g_prime[:1]+g_prime[1:2]),
        )
        self._set_params()
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(laplacian(psi,dx,dy) - beta_plane.f0**2 * (1/H1/g1+1/H1/g2)*psi[...,1:-1,1:-1]) + beta_effect
    
    def setup(self, psis: list[torch.Tensor],times:list[torch.Tensor],beta_effect_w:torch.Tensor,sf0_da:xr.DataArray) -> None:
        psi0_filt_da = xr.apply_ufunc(
            gaussian_filter,
            sf0_da.load(),
            kwargs={"sigma": self.sigma},
            input_core_dims=[["i", "j"]],
            output_core_dims=[["i", "j"]],
            vectorize=True,
        )
        psi0 = (
            torch.tensor(psi_regridder(psi0_filt_da).data, **specs)
            .unsqueeze(0)
            .unsqueeze(0)
            / beta_plane.f0
        )
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
            Boundaries.extract(
                self.compute_q(psi[:, :1],beta_effect_w), 2, -3, 2, -3, 3
            )
            for psi in psis
        ]
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        self.model.set_psiq(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],beta_effect_w),b-1))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

### ψ₂ transported

In [ ]:
from qgsw.decomposition.coefficients import DecompositionCoefs
from qgsw.models.qg.psiq.modified.forced import QGPSIQRGPsi2TransportDR
from qgsw.models.qg.stretching_matrix import compute_A_tilde


class SurfML(ModelWrapperOBC[QGPSIQRGPsi2TransportDR]):
    prefix = "results_enatl60"
    color="navy"
    label="GaussBarotropic - RG - DR"
    save_video = False
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQRGPsi2TransportDR(
            space_2d=space_2d,
            H=H[:2],
            beta_plane=beta_plane,
            g_prime=g_prime[:2],
        )
        self._set_params()
        self.alphas = {}
        self.coefs = {}
    def compute_q(self,psi: Tensor, A11:torch.Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi, dx, dy)
            - beta_plane.f0**2 * A11 * psi[..., 1:-1, 1:-1]
        ) + beta_effect
    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor,sf0_da:xr.DataArray) -> None:
        res = self.load()

        try:
            alpha:torch.Tensor = res[self.cycle]["alpha"]
        except KeyError:
            alpha=torch.tensor(0,**specs)
        try:
            self.uv10_to_uvsurf = res[self.cycle]["uv10_to_uvsurf"]
        except KeyError:
            ...
        self.A = compute_A_tilde(H[:2],g_prime[:2],alpha,**specs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))

        basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        try:
            basis.freeze_time_normalization(self.model.dt*torch.tensor([n_steps_per_cyle],**specs))
        except:... 
        basis.set_coefs(coefs)
        self._fpsi2 = basis.localize(
            space_interior.psi.xy.x,space_interior.psi.xy.y
        )

        if self.save_params:
            self.alphas[self.cycle] = alpha
            self.coefs[self.cycle] = coefs
        
        try:
            sigma = res[self.cycle]["config"]["sigma_ic"]
        except KeyError:
            sigma = 14
        psi0_filt_da = xr.apply_ufunc(
            gaussian_filter,
            sf0_da.load(),
            kwargs={"sigma": sigma},
            input_core_dims=[["i", "j"]],
            output_core_dims=[["i", "j"]],
            vectorize=True,
        )
        psi0 = (
            torch.tensor(psi_regridder(psi0_filt_da).data, **specs)
            .unsqueeze(0)
            .unsqueeze(0)
            / beta_plane.f0
        )
            
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
                Boundaries.extract(
                    self.compute_q(psi[:, :1],self.A[:1,:1],beta_effect_w), 2, -3, 2, -3, 3
                )
                for psi in psis
            ]

        self.model.set_psiq(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],self.A[:1,:1],beta_effect_w),b-1))
        self.model.alpha = alpha
        self.model.basis = basis
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

### Forced

In [ ]:
from qgsw.models.qg.psiq.modified.forced import QGPSIQForced

Heq = H[:1]*H[1:2]/(H[:1]+H[1:2])
class VarDyn(ModelWrapperOBC[QGPSIQForced]):
    prefix = "results_enatl60_forced"
    color="brown"
    label="Forced DR"
    save_video = False
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQForced(
            space_2d=space_2d,
            H=Heq,
            beta_plane=beta_plane,
            g_prime=g_prime[1:2],
        )
        self.model.wind_scaling = H[:1].item()
        self._set_params()
        self.coefs = {}
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi,dx,dy)
            - beta_plane.f0**2 * (1/Heq/g2)*psi[...,1:-1,1:-1]
        ) + beta_effect

    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor,sf0_da:xr.DataArray) -> None:
        res = self.load()
        try:
            self.uv10_to_uvsurf = res[self.cycle]["uv10_to_uvsurf"]
        except KeyError:
            ...
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))
        self.basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        self.basis.set_coefs(coefs)
        if self.save_params:
            self.coefs[self.cycle] = coefs
            
        self.wv = self.basis.localize(
            self.model.space.remove_h().q.xy.x,
            self.model.space.remove_h().q.xy.y,
        )
        
        
        try:
            sigma = res[self.cycle]["config"]["sigma_ic"]
        except KeyError:
            sigma = 14
        psi0_filt_da = xr.apply_ufunc(
            gaussian_filter,
            sf0_da.load(),
            kwargs={"sigma": sigma},
            input_core_dims=[["i", "j"]],
            output_core_dims=[["i", "j"]],
            vectorize=True,
        )
        psi0 = (
            torch.tensor(psi_regridder(psi0_filt_da).data, **specs)
            .unsqueeze(0)
            .unsqueeze(0)
            / beta_plane.f0
        )
            
        
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
                Boundaries.extract(
                    self.compute_q(psi[:, :1],beta_effect_w), 2, -3, 2, -3, 3
                )
                for psi in psis
            ]

        self.model.set_psiq(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],beta_effect_w),b-1))
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())
        
    def step(self) -> None:
        self.model.forcing = self.wv(self.model.time)
        super().step()

### Forced + SurfML

In [ ]:
from qgsw.models.qg.psiq.modified.forced import QGPSIQForcedSurfML

class ForcedSurfML(ModelWrapperOBC[QGPSIQForcedSurfML]):
    prefix = "results_enatl60_forcedsurf"
    color="coral"
    label="Forced + SurfML"
    save_video = False
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQForcedSurfML(
            space_2d=space_2d,
            H=H[:2],
            beta_plane=beta_plane,
            g_prime=g_prime[:2],
        )
        self._set_params()
        self.coefs = {}
        self.alphas = {}
        self.coefs_wv={}
    def compute_q(self,psi: Tensor, A11:torch.Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi, dx, dy)
            - beta_plane.f0**2 * A11 * psi[..., 1:-1, 1:-1]
        ) + beta_effect

    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor,sf0_da:xr.DataArray) -> None:
        res = self.load()

        try:
            alpha:torch.Tensor = res[self.cycle]["alpha"]
        except KeyError:
            alpha=torch.tensor(0,**specs)
        try:
            self.uv10_to_uvsurf = res[self.cycle]["uv10_to_uvsurf"]
        except KeyError:
            ...
        
        self.A = compute_A_tilde(H[:2],g_prime[:2],alpha,**specs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))
        coefs_wv = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs_wv"],**specs))

        basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        self.wv = build_basis_from_params_dict(res[self.cycle]["config"]["wv"])
        try:
            basis.freeze_time_normalization(self.model.dt*torch.tensor([n_steps_per_cyle],**specs))
        except:... 
        basis.set_coefs(coefs)
        self.wv.set_coefs(coefs_wv)
        self._fpsi2 = basis.localize(
            space_interior.psi.xy.x,space_interior.psi.xy.y
        )

        if self.save_params:
            self.alphas[self.cycle] = alpha
            self.coefs[self.cycle] = coefs
            self.coefs_wv[self.cycle] = coefs_wv
        self.fwv = self.wv.localize(
            self.model.space.remove_h().q.xy.x,
            self.model.space.remove_h().q.xy.y,
        )
        
        
        try:
            sigma = res[self.cycle]["config"]["sigma_ic"]
        except KeyError:
            sigma = 14
        psi0_filt_da = xr.apply_ufunc(
            gaussian_filter,
            sf0_da.load(),
            kwargs={"sigma": sigma},
            input_core_dims=[["i", "j"]],
            output_core_dims=[["i", "j"]],
            vectorize=True,
        )
        psi0 = (
            torch.tensor(psi_regridder(psi0_filt_da).data, **specs)
            .unsqueeze(0)
            .unsqueeze(0)
            / beta_plane.f0
        )
        
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
                Boundaries.extract(
                    self.compute_q(psi[:, :1],self.A[:1,:1],beta_effect_w), 2, -3, 2, -3, 3
                )
                for psi in psis
            ]

        self.model.set_psiq(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],self.A[:1,:1],beta_effect_w),b-1))
        self.model.alpha = alpha
        self.model.basis = basis
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())
        
    def step(self) -> None:
        self.model.forcing = self.fwv(self.model.time)
        super().step()

## Videos

In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.artist import Artist
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.image import AxesImage
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

from qgsw import plots
from qgsw.logging.utils import sec2text
from qgsw.plots.plt_wrapper import retrieve_imshow_data

def build_figure(psis_ref:list[torch.Tensor], times:np.ndarray, *mws:ModelWrapperOBC[T],retrieve_data:Callable[[ModelWrapperOBC],list[torch.Tensor]]=lambda mw, i : mw.psis[i][0,0], robust:bool=False) -> Figure:


    fig = plt.figure(figsize=((len(mws)+1)*3,10))
    gs = GridSpec(2,(len(mws)+1),figure=fig, width_ratios=[1]*len(mws)+[1.075])
    ax_ref = fig.add_subplot(gs[0,0])

    axs = [fig.add_subplot(gs[0,i+1]) for i in range(len(mws))]

    ax_l = fig.add_subplot(gs[1,:])
    title = plots.blittable_suptitle(sec2text(times[0]*3600*24),fig,ax_ref)
    if robust :
        vmax = max(crop(p[0,0],b).abs().quantile(0.98) for p in psis_ref)
    else:
        vmax = max(crop(p[0,0],b).abs().max() for p in psis_ref)

    im_ref = plots.imshow(crop(psis_ref[-1][0,0],b),vmax=vmax,vmin=-vmax,ax=ax_ref,show_cbar=False, title="Reference (non filtered)")
    divider = make_axes_locatable(axs[-1])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im_ref, cax=cax)
    ims :list[AxesImage]= []
    for i,mw in enumerate(mws):
        ims.append(plots.imshow(retrieve_data(mw,0),vmax=vmax,vmin=-vmax,ax=axs[i],show_cbar=False, title = mw.label, cbar_kwargs=dict(extendrect=not robust)))
    

    y_max = min(max(np.array(mw.losses["rmse"][-1]).max() for mw in mws),1)
    x_m,y_m = ax_l.margins()

    x_len = (times[-1]-times[0])
    y_len = y_max-0

    ax_l.set_xlim(times[0]-x_len*x_m, times[-1] + x_len*x_m)
    ax_l.set_ylim(0-y_len*y_m, y_max+y_len*y_m)

    losses:list[Line2D] = []
    dummy_loss = np.ones_like(np.array(mw.losses["rmse"][-1]))*np.nan
    for mw in mws:
        losses += ax_l.plot( times, dummy_loss ,**mw.plot_kwargs)

    for ax in axs:
        ax.set_yticks([])

    ax_l.legend(loc="upper left",prop={'size': 8})
    ax_l.set_xlabel("Time [days]")
    ax_l.set_ylabel("Normalized RMSE")
    
    def update(frame:int) -> tuple[Artist,...]:
        title.set_text(sec2text(times[frame]*3600*24))
        im_ref.set_array(retrieve_imshow_data(crop(psis_ref[frame][0,0],b)))
        for im, mw in zip(ims, mws):
            im.set_array(retrieve_imshow_data(retrieve_data(mw,frame)))
        
        for loss,mw in zip(losses,mws):
            vals = dummy_loss.copy()
            vals[:frame] = np.array(mw.losses["rmse"][-1][:frame])
            loss.set_data(times,vals)
        return title, im_ref, *ims, *losses
    return fig, update

def make_psi_video(filename:Path|str, psis_ref:list[torch.Tensor], times:np.ndarray, *mws: ModelWrapperOBC[T])-> None:
    filename = Path(filename)

    if not all(mw.remember_psis for mw in mws):
        return

    retrieve_data =lambda mw,i : mw.psis[i][0,0]

    fig, update = build_figure(psis_ref, times, *mws, retrieve_data=retrieve_data)

    FuncAnimation(fig, update, n_steps_per_cyle,blit=True).save(filename,fps=10)
    plt.close()

def make_vorticity_video(filename:Path|str, vorts_ref:list[torch.Tensor], times:np.ndarray, *mws: ModelWrapperOBC[T], **kwargs)-> None:
    filename = Path(filename)

    if not all(mw.remember_psis for mw in mws):
        return

    retrieve_data =lambda mw,i : laplacian(mw.psis[i][0,0],dx,dy)/beta_plane.f0
    kwargs.setdefault("robust",True)
    fig, update = build_figure(vorts_ref, times, *mws, retrieve_data=retrieve_data, **kwargs)

    FuncAnimation(fig, update, n_steps_per_cyle,blit=True).save(filename,fps=10)
    plt.close()



In [ ]:
def perform_cycle(season:str,c:int, models:ModelsManagerOBC) -> list[torch.Tensor]:
    in_season = retrieve_dates(*files.tolist()).month.isin(season_dict[season])
    if ((in_season[1:]) & (~in_season[:-1])).sum() + int(in_season[0]) > 1:
        msg = "Non-time-contiguous data for this season in provided dataset."
        raise ValueError(msg)
    season_files = files[in_season]
    season_ufiles = ufiles[in_season]
    season_vfiles = vfiles[in_season]

    models.new_cycle()

    psis_ref,psis_filt, times,sf0_da=load_cycle_data_atmp(season_files,c)

    u_refs, v_refs = load_cycle_uv_data(season_ufiles, season_vfiles,c)

    models.reset_time()

    models.setup(psis_filt,times,beta_effect[:,1:-1], sf0_da)

    models.compute_loss(crop(psis_ref[0],b))
    models.compute_uv_losses(crop(u_refs[0],b),crop(v_refs[0],b))
    models.compute_psi_filt_loss(crop(psis_filt[0],b))
    for n in range(1,n_steps_per_cyle):
        models.step()
        models.compute_loss(crop(psis_ref[n],b))
        models.compute_uv_losses(crop(u_refs[n],b),crop(v_refs[n],b))
        models.compute_psi_filt_loss(crop(psis_filt[n],b))
    return psis_ref


In [ ]:
def main(season:str, *wrappers:ModelWrapperOBC[QGPSIQCore]) -> tuple[ModelsManagerOBC[QGPSIQCore],list[torch.Tensor]]:

    in_season = retrieve_dates(*files.tolist()).month.isin(season_dict[season])
    if ((in_season[1:]) & (~in_season[:-1])).sum() + int(in_season[0]) > 1:
        msg = "Non-time-contiguous data for this season in provided dataset."
        raise ValueError(msg)
    season_vort_files = vort_files[in_season]

    models = ModelsManagerOBC(
        *wrappers
    )
    models.save_params = True

    models.remember_psis(True)

    gc.collect()
    for c in range(n_cycles):
        torch.cuda.reset_peak_memory_stats()
        psis_ref = perform_cycle(season,c,models)

        torch.cuda.empty_cache()
        gc.collect()

        max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
        msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
        logger.info(box(msg_mem, style="="))
    
    return models, psis_ref


In [ ]:
space_interior.ny

In [ ]:
beta_plane

In [ ]:
from qgsw.observations.satellite_track import SatelliteTrackMask


obs_mask = SatelliteTrackMask(
    space_interior.psi.xy.x,
    space_interior.psi.xy.y,
    track_width=100000,
    track_interval=550000,
    theta=torch.pi / 12,
    full_coverage_time=20 * 3600 * 24,
)
print(obs_mask.compute_obs_nb(240,7200))
obs_nb = [torch.sum(obs_mask.at_time(torch.tensor([i*7200],**specs))) for i in range(240)]

In [ ]:
plt.scatter([np.array(i*7200) for i in range(240)],[e.cpu().numpy() for e in obs_nb])

## Summer

In [ ]:
rg = ReducedGravity(space_interior)


forced_g100000 = VarDyn(space_interior)
forced_g100000.prefix = "results_enatl60_forced_atmp_gamma100000_nowind_obstrack_summer"
forced_g100000.linestyle = "solid"
forced_g100000.label = "Forced - R"

surfml_g0_1 = SurfML(space_interior)
surfml_g0_1.prefix = "results_enatl60_atmp_gamma0_1_nowind_obstrack_summer"
surfml_g0_1.color = "navy"
surfml_g0_1.linestyle = "solid"
surfml_g0_1.label = "SurfML - R"

surfml_noreg = SurfML(space_interior)
surfml_noreg.prefix = "results_enatl60_atmp_noreg_nowind_obstrack_summer"
surfml_noreg.color = "navy"
surfml_noreg.linestyle = "dashed"
surfml_noreg.label = "SurfML - NR"

season = "summer"

in_season = retrieve_dates(*files.tolist()).month.isin(season_dict[season])
if ((in_season[1:]) & (~in_season[:-1])).sum() + int(in_season[0]) > 1:
    msg = "Non-time-contiguous data for this season in provided dataset."
    raise ValueError(msg)
season_vort_files = vort_files[in_season]

models = ModelsManagerOBC(
    rg,forced_g100000, surfml_noreg, surfml_g0_1,
)
models.save_params = True

models.remember_psis(True)

gc.collect()
for c in range(n_cycles):

    times = np.arange(n_steps_per_cyle)*dt+c*n_steps_per_cyle*dt

    torch.cuda.reset_peak_memory_stats()
    psis_ref = perform_cycle(season,c,models)

    vorticities = load_cycle_vorticity_data(season_vort_files, c)

    make_vorticity_video(f"../output/videos/eNATL60/{season}/vorticity_atmp_c{c}.mp4", vorticities, times/3600/24, rg, forced_g100000, surfml_g0_1, robust=True)

    make_psi_video(f"../output/videos/eNATL60/{season}/psi_atmp_c{c}.mp4",psis_ref,times/3600/24,rg,forced_g100000,surfml_g0_1)

    torch.cuda.empty_cache()
    gc.collect()

    max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
    msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
    logger.info(box(msg_mem, style="="))


In [ ]:
from matplotlib import pyplot as plt

from qgsw import plots

# To Show
forced_g100000.linestyle = "solid"
# Plots

show_filt = True
show_grad = True

fig,axs = plots.subplots(1+show_filt+show_grad,1,figsize=(22,(5)*(1+show_filt+show_grad)))
fig.suptitle(season.capitalize())
plots.set_rowtitles(["RMSE"]+show_filt*[ "Filtered Stream function RMSE"]+show_grad*[ "Gradient RMSE"] ,axs=axs)
models.plot_loss(loss_name="rmse",ax=axs[0,0])
plots.clamp_ylims(0,1,axs[0,0])
axs[0,0].legend(loc="upper left",prop={'size': 8})
if show_filt:
    models.plot_loss(loss_name="psi_filt_rmse",ax=axs[1,0])
    plots.clamp_ylims(0,1,axs[1,0])
    axs[1,0].legend(loc="upper left",prop={'size': 8})
if show_grad:
    models.plot_loss(loss_name="uv_rmse",ax=axs[show_filt+1,0])
    plots.clamp_ylims(0,1,axs[show_filt+1,0])
    axs[show_filt+1,0].legend(loc="upper left",prop={'size': 8})

In [ ]:
forced_g100000.label = "VarDyn"
surfml_g0_1.label = "SurfML"

times = np.arange(n_steps_per_cyle)*dt

sns.set_theme("notebook",style="dark")

fig, axs = plots.subplots(1,1, figsize=(10,5), dpi=150)
for mw in [rg, forced_g100000, surfml_g0_1]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between(times/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.2, color=mw.color)
    axs[0,0].plot(times/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend(fontsize="x-large")
axs[0,0].set_xlabel('Time [days]',fontsize="large")
axs[0,0].set_ylabel("nRMSE",fontsize="large")
axs[0,0].set_title(f"eNATL60 - {season.capitalize()}",fontsize = "xx-large")
plots.set_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/enatl60_rmse_{season}.svg')

fig, axs = plots.subplots(1,1, figsize=(10,5), dpi=150)
for mw in [rg, forced_g100000, surfml_g0_1]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["uv_rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between(times/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.2, color=mw.color)
    axs[0,0].plot(times/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend(fontsize="x-large")
axs[0,0].set_xlabel('Time [days]', fontsize="large")
axs[0,0].set_ylabel("Velocity RMSE", fontsize="large")
axs[0,0].set_title(f"eNATL60 - {season.capitalize()}",fontsize = "xx-large")
plots.set_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/enatl60_uv_rmse_{season}.svg')

In [ ]:
from copy import deepcopy


losses_summer = {
    "rg": deepcopy(rg.losses),
    "vardyn": deepcopy(forced_g100000.losses),
    "surfml": deepcopy(surfml_g0_1.losses),
}

## Winter

In [ ]:
rg = ReducedGravity(space_interior)


forced_g100000 = VarDyn(space_interior)
forced_g100000.prefix = "results_enatl60_forced_atmp_gamma100000_nowind_obstrack_winter"
forced_g100000.linestyle = "solid"
forced_g100000.label = "Forced - R"

surfml_g0_1 = SurfML(space_interior)
surfml_g0_1.prefix = "results_enatl60_atmp_gamma0_1_nowind_obstrack_winter"
surfml_g0_1.color = "navy"
surfml_g0_1.linestyle = "solid"
surfml_g0_1.label = "SurfML - R"

surfml_noreg = SurfML(space_interior)
surfml_noreg.prefix = "results_enatl60_atmp_noreg_nowind_obstrack_winter"
surfml_noreg.color = "navy"
surfml_noreg.linestyle = "dashed"
surfml_noreg.label = "SurfML - NR"

season = "winter"

in_season = retrieve_dates(*files.tolist()).month.isin(season_dict[season])
if ((in_season[1:]) & (~in_season[:-1])).sum() + int(in_season[0]) > 1:
    msg = "Non-time-contiguous data for this season in provided dataset."
    raise ValueError(msg)
season_vort_files = vort_files[in_season]

models = ModelsManagerOBC(
    rg,forced_g100000, surfml_noreg, surfml_g0_1,
)
models.save_params = True

models.remember_psis(True)

gc.collect()
for c in range(n_cycles):

    times = np.arange(n_steps_per_cyle)*dt+c*n_steps_per_cyle*dt

    torch.cuda.reset_peak_memory_stats()
    psis_ref = perform_cycle(season,c,models)

    vorticities = load_cycle_vorticity_data(season_vort_files, c)

    make_vorticity_video(f"../output/videos/eNATL60/{season}/vorticity_atmp_c{c}.mp4", vorticities, times/3600/24, rg, forced_g100000, surfml_g0_1, robust=True)

    make_psi_video(f"../output/videos/eNATL60/{season}/psi_atmp_c{c}.mp4",psis_ref,times/3600/24,rg,forced_g100000,surfml_g0_1)

    torch.cuda.empty_cache()
    gc.collect()

    max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
    msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
    logger.info(box(msg_mem, style="="))


In [ ]:
from matplotlib import pyplot as plt

from qgsw import plots

# To Show
forced_g100000.linestyle = "solid"
# Plots

show_filt = True
show_grad = True

fig,axs = plots.subplots(1+show_filt+show_grad,1,figsize=(22,(5)*(1+show_filt+show_grad)))
fig.suptitle("Winter")
plots.set_rowtitles(["RMSE"]+show_filt*[ "Filtered Stream function RMSE"]+show_grad*[ "Gradient RMSE"] ,axs=axs)
models.plot_loss(loss_name="rmse",ax=axs[0,0])
plots.clamp_ylims(0,1,axs[0,0])
axs[0,0].legend(loc="upper left",prop={'size': 8})
if show_filt:
    models.plot_loss(loss_name="psi_filt_rmse",ax=axs[1,0])
    plots.clamp_ylims(0,1,axs[1,0])
    axs[1,0].legend(loc="upper left",prop={'size': 8})
if show_grad:
    models.plot_loss(loss_name="uv_rmse",ax=axs[show_filt+1,0])
    plots.clamp_ylims(0,1,axs[show_filt+1,0])
    axs[show_filt+1,0].legend(loc="upper left",prop={'size': 8})

In [ ]:
forced_g100000.label = "VarDyn"
surfml_g0_1.label = "SurfML"

times = np.arange(n_steps_per_cyle)*dt

sns.set_theme("notebook",style="dark")

fig, axs = plots.subplots(1,1, figsize=(10,10/gratio), dpi=150)
for mw in [rg, forced_g100000, surfml_g0_1]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between(times/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.2, color=mw.color)
    axs[0,0].plot(times/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend(fontsize="xx-large")
axs[0,0].set_xlabel('Time [days]',fontsize="x-large")
axs[0,0].set_ylabel("nRMSE",fontsize="x-large")
axs[0,0].set_title(f"eNATL60 - {season.capitalize()}",fontsize = "xx-large")
plots.set_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/enatl60_rmse_{season}.svg')

fig, axs = plots.subplots(1,1, figsize=(10,10/gratio), dpi=150)
for mw in [rg, forced_g100000, surfml_g0_1]:
    ls = torch.stack([torch.tensor(l,**specs) for l in mw.losses["uv_rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    axs[0,0].fill_between(times/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.2, color=mw.color)
    axs[0,0].plot(times/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)
axs[0,0].legend(fontsize="x-large")
axs[0,0].set_xlabel('Time [days]', fontsize="large")
axs[0,0].set_ylabel("Velocity RMSE", fontsize="large")
axs[0,0].set_title(f"eNATL60 - {season.capitalize()}",fontsize = "xx-large")
plots.set_ylims(0,1,axs[0,0])
plots.show()
fig.savefig(f'../output/images/enatl60_uv_rmse_{season}.svg')

In [ ]:
from copy import deepcopy


losses_winter = {
    "rg": deepcopy(rg.losses),
    "vardyn": deepcopy(forced_g100000.losses),
    "surfml": deepcopy(surfml_g0_1.losses),
}

In [ ]:
loss1 = loss2 = np.array(surfml_g0_1.losses["rmse"][0])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import Size, Divider
import matplotlib.ticker as ticker

fig = plt.figure(dpi=150)

# Define fixed sizes in inches
pad_left   = Size.Fixed(0.8)   # space for y-tick labels on the left
ax_width   = Size.Fixed(5.0)   # plotting area width — same for both!
gap        = Size.Fixed(0.1)   # gap between panels
pad_right  = Size.Fixed(0.3)   # right margin

h = [pad_left, ax_width, gap, ax_width, pad_right]
v = [Size.Fixed(0.6), Size.Fixed(5/gratio), Size.Fixed(0.4)]  # bottom, height, top

divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)

ax1 = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=1))
ax2 = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=1),
                   sharey=ax1)

times = np.arange(n_steps_per_cyle)*dt

for mw, loss_name in zip([rg, forced_g100000, surfml_g0_1],['rg',"vardyn","surfml"]):
    ls = torch.stack([torch.tensor(l,**specs) for l in losses_summer[loss_name]["rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    ax1.fill_between(times/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.2, color=mw.color)
    ax1.plot(times/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)

for mw, loss_name in zip([rg, forced_g100000, surfml_g0_1],['rg',"vardyn","surfml"]):
    ls = torch.stack([torch.tensor(l,**specs) for l in losses_winter[loss_name]["rmse"]])
    ms = torch.mean(ls,dim=0)
    stds = torch.std(ls,dim=0)
    ax2.fill_between(times/3600/24, np.subtract(ms.cpu().numpy(), stds.cpu().numpy()), np.add(ms.cpu().numpy(), stds.cpu().numpy()), alpha=0.2, color=mw.color)
    ax2.plot(times/3600/24,ms.cpu().numpy(), **mw.plot_kwargs)

# fig.suptitle("Three-Layer QG")

ax1.legend(fontsize="large")

ax1.set_title("Summer", fontsize="x-large")
ax2.set_title("Winter", fontsize="x-large")

ax1.set_xlabel("Time [days]", fontsize="large")
ax1.set_ylabel("nRMSE", fontsize="large")
ax2.set_xlabel("Time [days]", fontsize="large")

ax1.tick_params(axis='both', labelsize="large")
ax1.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.0f'))
ax2.tick_params(axis='both', labelsize="large")
ax2.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.0f'))

ax2.tick_params(labelleft=False)  # hide right panel y-labels
# plt.tight_layout()
plt.savefig("../output/images/enatl60_rmses.svg",bbox_inches="tight")

### Vorticity plots

In [ ]:
surfml_g0_1 = SurfML(space_interior)
surfml_g0_1.prefix = "results_enatl60_atmp_gamma0_1_nowind_obstrack_summer"
surfml_g0_1.color = "navy"
surfml_g0_1.linestyle = "solid"
surfml_g0_1.label = "SurfML - R"

season = "summer"

in_season = retrieve_dates(*files.tolist()).month.isin(season_dict[season])
if ((in_season[1:]) & (~in_season[:-1])).sum() + int(in_season[0]) > 1:
    msg = "Non-time-contiguous data for this season in provided dataset."
    raise ValueError(msg)
season_vort_files = vort_files[in_season]

models = ModelsManagerOBC(
    surfml_g0_1,
)
models.save_params = True

models.remember_psis(True)

cycle = 0

gc.collect()

times = np.arange(n_steps_per_cyle)*dt+cycle*n_steps_per_cyle*dt

torch.cuda.reset_peak_memory_stats()
psis_ref = perform_cycle(season,cycle,models)

vorticities = load_cycle_vorticity_data(season_vort_files,  cycle)

torch.cuda.empty_cache()
gc.collect()

max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
msg_mem = f"Cycle {step(cycle + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
logger.info(box(msg_mem, style="="))

In [ ]:
steps = [60, 120, 180, 240]

fig, axs = plots.subplots(2,len(steps),gridspec_kw={"width_ratios":[1,1,1,1]}, figsize=(4*len(steps),10.5),dpi=150)

dvort = load_datasets(vort_files[0], format_func=format_dvort)
ds_lon = dvort[LONGITUDE]
ds_lat=dvort[LATITUDE]
lon_slice = (ds_lon >= np.rad2deg(lons[(b+1):-(b+1),(b+1):-(b+1)].min())) & (ds_lon <= np.rad2deg(lons[(b+1):-(b+1),(b+1):-(b+1)].max()))
lat_slice = (ds_lat >= np.rad2deg(lats[(b+1):-(b+1),(b+1):-(b+1)].min())) & (ds_lat <= np.rad2deg(lats[(b+1):-(b+1),(b+1):-(b+1)].max()))

Is,Js = np.meshgrid(dvort.i.values,dvort.j.values,indexing="ij")

I_slice = Is[lon_slice&lat_slice]
J_slice = Js[lon_slice&lat_slice]

imin,imax = I_slice.min(),I_slice.max()
jmin,jmax = J_slice.min(),J_slice.max()

lon = ds_lon.sel(i=slice(imin,imax+1),j=slice(jmin,jmax+1))
lat = ds_lat.sel(i=slice(imin,imax+1),j=slice(jmin,jmax+1))

plots.set_coltitles([sec2text(s*dt) for s in steps],axs,fontsize="xx-large")
plots.set_rowtitles(["SurfML", "eNATL60"],axs,fontsize="xx-large",pad=10)

fig.suptitle("Vorticity fields",fontsize="xx-large")

vmax = max(vorticities[s-1][0,0].abs().quantile(0.98)for s in steps)

for i,s in enumerate(steps):

    plots.imshow(laplacian(surfml_g0_1.psis[s-1][0,0],dx,dy)/beta_plane.f0,ax=axs[0,i],vmin=-vmax,vmax=vmax,show_cbar=i == (len(steps)-1), extent=[0,space_interior.nx*10,0,space_interior.ny*10])
    im = axs[1,i].pcolormesh(lon,lat,vorticities[s-1][0,0].cpu().numpy(),cmap="RdBu_r",vmin=-vmax,vmax=vmax,rasterized=True)

    if i !=0:
        axs[0,i].set_yticks([])
        axs[1,i].set_yticks([])
    axs[0,i].set_xlabel("X [km]",fontsize="large")
    axs[1,i].set_xlabel("Longitude [°]",fontsize="large")
        
axs[0,-1].cbar.set_label("Vorticity (normalized by f₀)",fontsize="x-large")
axs[0,0].set_ylabel("Y [km]",fontsize="large")
axs[1,0].set_ylabel("Latitude",fontsize="large")
div = make_axes_locatable(axs[1,-1])
cax = div.append_axes("right", size="5%", pad="3%")
cbar = plt.colorbar(im,cax=cax)
cbar.set_label("Vorticity (normalized by f₀)",fontsize="x-large")
for ax in axs.ravel():
    ax.set_aspect('auto')
plt.tight_layout()
fig.savefig(f"../output/images/enatl60_vorticity_{season}.svg")

In [ ]:
axs[0,0].set_ylabel("Latitude",fontsize="large")

In [ ]:
plt.show()